# FGD Multimodal Master's Thesis Pipeline — Colab (T4-safe, Drive-native)

Runs the full focus-group workflow against the recording stored in your Google Drive:

**Drive source → audio mastering → faster-whisper large-v3 transcription → optional PyAnnote speaker diarization → optional Qwen-VL frame captions → Qwen2.5-3B thematic analysis → thesis report → DOCX + ZIP saved on your Drive.**

Key properties:

1. **Nothing uploaded by hand.** The notebook mounts Drive, auto-detects the FGD recording inside `FGD_DIR` (preferred order: `.mp4`, `.mov`, `.mkv`, `.avi`, then `.mp3`, `.wav`, `.m4a`, `.aac`, `.flac`, `.ogg`, `.opus`), and reads it directly.
2. **Every artifact is written to your Drive** as it is produced, so partial work survives even if a later cell fails. Final package lives in `FGD_DIR/FGD_COMPLETE_OUTPUT/` and a ZIP is created there and offered for download.
3. **Speaker diarization is optional but automatic**: the notebook reads `HF_TOKEN` from Colab Secrets or from `HF_TOKEN.txt` inside `FGD_DIR`; if a token is present, PyAnnote speaker-diarization-3.1 labels every segment; otherwise segments keep `UNVERIFIED_SPEAKER` and the pipeline still completes.
4. **Stages are split into separate cells** so you can see where a failure happens and re-run only that stage.
5. Clear asserts (GPU present, source exists, speech detected) and VRAM cleanup between models.

**Before running:** `Runtime → Change runtime type → T4 GPU`. Then run cells **in order, top to bottom**.

*This notebook produces a machine-assisted working package. Every transcript, speaker label, and theme must be reviewed by the researcher before any thesis claim is made.*

### Step 0 — Mount Drive, check GPU, load the Hugging Face token

In [ ]:
from google.colab import drive, userdata
import os, sys, torch

drive.mount('/content/drive', force_remount=True)

# --- GPU check ---------------------------------------------------------------
print(f"Python: {sys.version.split()[0]}")
assert torch.cuda.is_available(), (
    "No GPU visible. Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU, "
    "then re-run this cell."
)
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)

# --- HF token (for optional PyAnnote diarization) ----------------------------
# Priority: Colab Secrets (HF_TOKEN) > HF_TOKEN.txt inside the FGD Drive folder.
hf_token = None
try:
    hf_token = userdata.get('HF_TOKEN') or None
except Exception:
    pass

if hf_token is None:
    token_file = "/content/drive/MyDrive/FGD/HF_TOKEN.txt"
    if os.path.exists(token_file):
        hf_token = open(token_file, encoding="utf-8").read().strip() or None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF token loaded (from Colab Secrets or HF_TOKEN.txt). Diarization will be attempted.")
else:
    print("No HF token found. Diarization will be skipped; speaker labels stay UNVERIFIED.")

### Step 1 — Install only what's missing

We deliberately do **not** touch `torch`/`torchvision`/`torchaudio` — Colab's T4 runtime ships a CUDA-matched build already, and reinstalling it was what caused earlier hangs. `ffmpeg` is already present on Colab too; the `apt-get` call is a safety net.

In [ ]:
%%capture install_log
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -q -U "transformers>=4.49.0" accelerate bitsandbytes
!pip install -q -U faster-whisper librosa soundfile python-docx qwen-vl-utils
!pip install -q pyannote.audio

In [ ]:
# Sanity check the installs actually landed.
import importlib
for pkg in ["transformers", "accelerate", "bitsandbytes", "faster_whisper",
            "librosa", "soundfile", "docx", "qwen_vl_utils", "pyannote.audio"]:
    mod = importlib.import_module(pkg)
    print(f"{pkg:22s} OK  ({getattr(mod, '__version__', 'n/a')})")

### Step 2 — Auto-detect the FGD source on Drive and create the Drive output folder

In [ ]:
from pathlib import Path
import subprocess, json

# --- Edit only these two variables ------------------------------------------
FGD_DIR = Path("/content/drive/MyDrive/FGD")     # Drive folder with the recording
PREFERRED = ".mp4"                                 # ".mp4" | ".mp3" | None (auto)
# -----------------------------------------------------------------------------

assert FGD_DIR.exists(), f"Drive folder not found: {FGD_DIR} (is Drive mounted?)"

VIDEO_EXT = {".mp4", ".mov", ".mkv", ".avi"}
AUDIO_EXT = {".mp3", ".wav", ".m4a", ".aac", ".flac", ".ogg", ".opus"}
ALL_EXT = VIDEO_EXT | AUDIO_EXT

# Collect candidate recordings (skip the HF token note and the output folder itself).
cands = [
    p for p in FGD_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in ALL_EXT
       and "HF_TOKEN" not in p.name and "FGD_COMPLETE_OUTPUT" not in str(p.parent)
]
assert cands, f"No supported media file found in {FGD_DIR}"

# Prefer the chosen extension, then any video, then any audio.
def rank(p):
    s = p.suffix.lower()
    return (0 if PREFERRED and s == PREFERRED else 1,
            0 if s in VIDEO_EXT else 1,
            -p.stat().st_size)

SOURCE = sorted(cands, key=rank)[0]
HAS_VIDEO = SOURCE.suffix.lower() in VIDEO_EXT
print("Source selected:", SOURCE.name, "| contains video:", HAS_VIDEO)

DRIVE_OUT = FGD_DIR / "FGD_COMPLETE_OUTPUT"
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/fgd_thesis")
WORK.mkdir(exist_ok=True)
print("Output Drive folder:", DRIVE_OUT)

### Step 3 — Master and compact the audio

Compacts long silences (>1.2 s reduced to 0.35 s) while preserving speech boundaries, denoises gently, and normalises to −16 LUFS. Timestamps in the transcript refer to this compacted `audio.wav`, not the original file.

In [ ]:
AUDIO = "/content/audio.wav"
FILTER = (
    "highpass=f=70,lowpass=f=7600,afftdn=nr=10:nf=-35:tn=1,"
    "acompressor=threshold=-20dB:ratio=2.5:attack=20:release=250,"
    "silenceremove=stop_periods=-1:stop_duration=1.2:stop_threshold=-38dB:stop_silence=0.35,"
    "loudnorm=I=-16:TP=-1.5:LRA=11"
)

subprocess.run(
    ["ffmpeg", "-hide_banner", "-y", "-i", str(SOURCE), "-vn", "-af", FILTER,
     "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", AUDIO],
    check=True,
)

probe = json.loads(subprocess.check_output(
    ["ffprobe", "-v", "error", "-show_entries",
     "format=duration,size:stream=codec_type,codec_name,width,height,sample_rate,channels",
     "-of", "json", str(SOURCE)], text=True))
audio_probe = json.loads(subprocess.check_output(
    ["ffprobe", "-v", "error", "-show_entries",
     "format=duration,size:stream=sample_rate,channels", "-of", "json", AUDIO], text=True))

processing = {
    "source": str(SOURCE),
    "source_probe": probe,
    "audio_output": "audio.wav",
    "audio_probe": audio_probe,
    "audio_processing": (
        "70 Hz high-pass; 7.6 kHz low-pass; gentle denoise; compression; "
        "-16 LUFS target; -1.5 dBTP target; silences longer than 1.2s reduced to 0.35s."
    ),
    "timestamp_rule": "Transcript times map to the compacted audio.wav, not the original video.",
}
(DRIVE_OUT / "processing_audit_trail.json").write_text(
    json.dumps(processing, indent=2), encoding="utf-8")
print(json.dumps(processing, indent=2))

### Step 4 — Transcribe with faster-whisper (large-v3, float16 on T4)

In [ ]:
import gc, torch, pandas as pd
from faster_whisper import WhisperModel

print("Loading Whisper large-v3 onto the GPU...")
whisper = WhisperModel("large-v3", device="cuda", compute_type="float16")

print("Transcribing...")
segs, info = whisper.transcribe(
    AUDIO, beam_size=5, best_of=5, temperature=0,
    word_timestamps=True, vad_filter=False, condition_on_previous_text=True,
)

rows = [{
    "segment_id": i,
    "compact_start": round(s.start, 2),
    "compact_end": round(s.end, 2),
    "duration": round(s.end - s.start, 2),
    "speaker": "UNVERIFIED_SPEAKER",
    "text": s.text.strip(),
    "avg_logprob": round(float(s.avg_logprob), 3),
} for i, s in enumerate(segs, 1)]

df = pd.DataFrame(rows)
assert not df.empty, "No speech was transcribed — check the source audio has audible speech."

def stamp(t):
    return f"{int(t // 3600):02}:{int(t % 3600 // 60):02}:{int(t % 60):02}"

transcript = "\n".join(
    f"[{stamp(r.compact_start)}–{stamp(r.compact_end)}] {r.speaker}: {r.text}"
    for r in df.itertuples()
)

(WORK / "transcript_timestamped.txt").write_text(transcript, encoding="utf-8")
df.to_csv(WORK / "transcription.csv", index=False, encoding="utf-8-sig")
# Persist to Drive immediately.
(WORK / "transcript_timestamped.txt").write_text(transcript, encoding="utf-8")
(DRIVE_OUT / "transcript_timestamped.txt").write_text(transcript, encoding="utf-8")
df.to_csv(DRIVE_OUT / "transcription.csv", index=False, encoding="utf-8-sig")

print("Language:", info.language, "| confidence:", round(info.language_probability, 3), "| segments:", len(df))
print(transcript[:1200])

del whisper
gc.collect()
torch.cuda.empty_cache()

### Step 5 — Optional speaker diarization (PyAnnote)

Uses `pyannote/speaker-diarization-3.1` with the HF token (Colab Secrets `HF_TOKEN` or `HF_TOKEN.txt` in the FGD folder). If no token is available, or the gated model is not accepted on Hugging Face, speaker labels remain `UNVERIFIED_SPEAKER`. **Diarization labels are machine-generated and must be verified by the researcher; they do not identify real names.**

In [ ]:
import torch

diarization_df = None
if hf_token:
    try:
        from pyannote.audio import Pipeline
        diar_pipe = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-3.1", use_auth_token=hf_token)
        if torch.cuda.is_available():
            diar_pipe.to(torch.device("cuda"))
        print("Running diarization (this can take a while for long recordings)...")
        diar = diar_pipe(AUDIO)
        turns = []
        for turn, _, speaker in diar.itertracks(yield_label=True):
            turns.append({"start": turn.start, "end": turn.end, "speaker": speaker})
        diarization_df = pd.DataFrame(turns)
        print("Diarization turns:", len(diarization_df))
    except Exception as exc:
        print("Diarization unavailable; keeping UNVERIFIED_SPEAKER labels.")
        print("Reason:", repr(exc)[:300])
        diarization_df = None

if diarization_df is not None and not diarization_df.empty:
    def get_speaker(start, end):
        cand = diarization_df[(diarization_df["start"] <= end) & (diarization_df["end"] >= start)]
        if cand.empty:
            return "UNVERIFIED_SPEAKER"

        def overlap(r):
            return min(end, r["end"]) - max(start, r["start"])

        best = cand.iloc[cand.apply(overlap, axis=1).values.argmax()]
        return str(best["speaker"])

    df["speaker"] = df.apply(lambda r: get_speaker(r["compact_start"], r["compact_end"]), axis=1)
    transcript = "\n".join(
        f"[{stamp(r.compact_start)}–{stamp(r.compact_end)}] {r['speaker']}: {r['text']}"
        for r in df.itertuples()
    )
    (DRIVE_OUT / "transcript_timestamped.txt").write_text(transcript, encoding="utf-8")
    df.to_csv(DRIVE_OUT / "transcription.csv", index=False, encoding="utf-8-sig")
    print("Speakers attached.", df["speaker"].nunique(), "distinct labels; first 20 rows:")
    print(df[["segment_id", "compact_start", "compact_end", "speaker"]].head(20).to_string(index=False))
else:
    print("No diarization applied.")

### Step 6 — Optional visual evidence (video sources only)

Samples up to 20 spread-out frames for observable interaction/context (not identity, emotion, or intent) and captions them with Qwen2.5-VL-3B in 4-bit. Skipped automatically for audio-only sources.

In [ ]:
import numpy as np, gc, torch, json

frames = []
if HAS_VIDEO:
    frame_dir = WORK / "video_frames"
    frame_dir.mkdir(exist_ok=True)
    duration = float(probe["format"]["duration"])
    times = np.linspace(0, duration, 20, endpoint=False) + min(5, duration / 40)
    for n, t in enumerate(times, 1):
        out = frame_dir / f"frame_{n:02d}_{int(t):04d}s.jpg"
        subprocess.run(
            ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-ss", str(round(float(t), 2)),
             "-i", str(SOURCE), "-frames:v", "1", "-vf", "scale='min(960,iw)':-2", str(out)],
            check=True,
        )
        frames.append({"original_video_time_seconds": round(float(t), 2), "file": str(out)})
print("Frames sampled:", len(frames))

visual = []
if frames:
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
    from qwen_vl_utils import process_vision_info

    qconf = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
    vl = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen2.5-VL-3B-Instruct", quantization_config=qconf, device_map="auto")
    vl_device = next(vl.parameters()).device

    visual_prompt = (
        "This is a sampled frame from a focus-group recording. Describe only plainly observable, "
        "research-relevant visual facts: setting, seating/visibility, materials, apparent turn-taking "
        "or group interaction, technical/recording issues. Do not identify people, infer demographics, "
        "emotions, intent, relationships, or sensitive traits. State uncertainty and write 3–6 short "
        "evidence-log bullets."
    )

    for item in frames:
        msg = [{"role": "user", "content": [
            {"type": "image", "image": item["file"]},
            {"type": "text", "text": visual_prompt},
        ]}]
        text = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        ii, vi = process_vision_info(msg)
        inp = processor(text=[text], images=ii, videos=vi, padding=True, return_tensors="pt").to(vl_device)
        ids = vl.generate(**inp, max_new_tokens=280, do_sample=False)
        item["observation"] = processor.batch_decode(ids[:, inp.input_ids.shape[1]:], skip_special_tokens=True)[0]
        visual.append(item)
        print("  frame", item["original_video_time_seconds"], "s ->", item["observation"][:100].replace(chr(10), " "))

    del vl, processor
    gc.collect()
    torch.cuda.empty_cache()

(DRIVE_OUT / "visual_evidence_log.json").write_text(
    json.dumps(visual, ensure_ascii=False, indent=2), encoding="utf-8")
print("Visual evidence entries:", len(visual))

### Step 7 — Audio technical quality notes

In [ ]:
import librosa, numpy as np, json

y, sr = librosa.load(AUDIO, sr=16000, mono=True)
voice_note = {
    "compact_duration_seconds": round(len(y) / sr, 2),
    "sample_rate": sr,
    "channels": 1,
    "mean_rms": round(float(np.sqrt(np.mean(y ** 2))), 5),
    "estimated_speech_segments": len(df),
    "interpretation": (
        "Technical recording indicators only. Do not treat these measures as participant "
        "traits, emotion, credibility, or diagnosis."
    ),
}
(DRIVE_OUT / "audio_voice_technical_notes.json").write_text(
    json.dumps(voice_note, indent=2), encoding="utf-8")
print(json.dumps(voice_note, indent=2))

### Step 8 — Transcript-grounded thematic analysis and thesis report (Qwen2.5-3B-Instruct, 4-bit)

Long transcripts are chunked; each chunk is analysed separately, then a single report is generated in two passes (sections 1–8 and 9–15) to stay inside T4 VRAM. The model is instructed to ground every claim in the transcript and never invent quotes, identity, or facts.

In [ ]:
import gc, torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

qconf = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
llm = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct", quantization_config=qconf, device_map="auto")
llm_device = next(llm.parameters()).device

def ask(prompt, n=1700):
    msgs = [
        {"role": "system", "content": (
            "You are a master's-level qualitative research assistant. Use supplied evidence only. "
            "Separate transcript evidence, machine observations, and researcher interpretation. "
            "Never invent quotations, speakers, identities, demographics, causality, consensus, "
            "or visual/emotional facts."
        )},
        {"role": "user", "content": prompt},
    ]
    inputs = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm_device)
    input_len = inputs["input_ids"].shape[1]
    z = llm.generate(**inputs, max_new_tokens=n, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(z[0][input_len:], skip_special_tokens=True)

def ask_safe(prompt, n):
    """Retry with a shrinking token budget instead of dying on one big OOM."""
    for attempt_n in [n, max(1000, n // 2), max(600, n // 3)]:
        gc.collect()
        torch.cuda.empty_cache()
        try:
            return ask(prompt, attempt_n)
        except torch.cuda.OutOfMemoryError:
            print(f"OOM at max_new_tokens={attempt_n}, retrying with a smaller budget...")
    raise RuntimeError("Still OOM even at a reduced token budget.")

lines = transcript.splitlines()
chunks, cur, length = [], [], 0
for line in lines:
    if length + len(line) > 8500 and cur:
        chunks.append("\n".join(cur))
        cur, length = [], 0
    cur.append(line)
    length += len(line) + 1
if cur:
    chunks.append("\n".join(cur))

print(f"Analyzing {len(chunks)} transcript section(s)...")
part = [
    ask(
        "Analyse this transcript section. Return: questions asked; verbatim-grounded answers; "
        "provisional themes/subthemes; agreement/disagreement; counterexamples; timestamped evidence; "
        "data-quality cautions.\n\n" + c
    )
    for c in chunks
]

visual_text = json.dumps(visual, ensure_ascii=False) if visual else "No video frames were supplied."

report_prompt = (
    "Create a comprehensive, thesis-ready Focus Group Discussion analytic report from the "
    "transcript analyses and visual evidence log below. Required headings:\n\n"
    "1. Study/Session Metadata (use placeholders where unknown)\n"
    "2. Data Sources and Processing Audit Trail\n"
    "3. Transcription and Speaker-Attribution Limitations\n"
    "4. Audio/Voice Technical Quality Notes\n"
    "5. Visual Evidence Log (only observable facts; mark machine-generated)\n"
    "6. Researcher Field Notes and Reflexive Notes Template\n"
    "7. Question\u2013Answer Matrix (question, evidence-based answer summary, dissent/caveat, timestamps)\n"
    "8. Thematic Analysis: theme, definition, subthemes, supporting evidence, negative case, interpretation\n"
    "9. Agreement/Disagreement and Interaction Patterns\n"
    "10. Preliminary Codebook (code, definition, include/exclude, example, related theme)\n"
    "11. Findings Narrative\n"
    "12. Trustworthiness: credibility, dependability, confirmability, transferability\n"
    "13. Ethics, anonymisation, and data-management notes\n"
    "14. Master's Thesis Reporting Checklist\n"
    "15. Limitations and Required Human Validation.\n\n"
    "Never present automated inference as fact. Do not name participants, infer traits, or "
    "claim video findings beyond the evidence log.\n\n"
    "TRANSCRIPT ANALYSES:\n" + "\n\n---\n".join(part) +
    "\n\nVISUAL EVIDENCE LOG:\n" + visual_text +
    "\n\nAUDIO TECHNICAL NOTE:\n" + json.dumps(voice_note)
)

report_prompt_1 = report_prompt + "\n\nFor this pass, write ONLY sections 1–8 of the required headings."
report_prompt_2 = report_prompt + "\n\nFor this pass, write ONLY sections 9–15 of the required headings."

part1 = ask_safe(report_prompt_1, 900)
gc.collect(); torch.cuda.empty_cache()
part2 = ask_safe(report_prompt_2, 900)

report = part1 + "\n\n" + part2
(DRIVE_OUT / "FGD_comprehensive_thesis_report.md").write_text(report, encoding="utf-8")
print(report[:1500])

del llm, tok
gc.collect()
torch.cuda.empty_cache()

### Step 9 — Package everything into a DOCX + ZIP on your Drive

The full package is written to `FGD_DIR/FGD_COMPLETE_OUTPUT/` on Google Drive, zipped, and offered for download. A run manifest records hashes of the source and of every artifact so the output is auditable and reproducible.

In [ ]:
import shutil, json, hashlib
from pathlib import Path
from docx import Document
from google.colab import files

# Copy local working artifacts (audio + video frames) into the Drive output.
for p in [Path(AUDIO)] + (list((WORK / "video_frames").glob("*.jpg")) if (WORK / "video_frames").exists() else []):
    if p.exists():
        shutil.copy(p, DRIVE_OUT / p.name)

# Researcher notes template.
researcher_template = '''# Researcher Notes and Validation Template
Session ID: ____  Date/time: ____  Facilitator: ____  Research question(s): ____

## Context and reflexivity
- Recruitment/context that may shape discussion:
- Facilitator actions or prompts:
- Researcher assumptions/positionality:
- Technical disruptions and their likely effect:

## Validation log
- Transcript passages corrected (timestamp / original / revised / reviewer):
- Speaker labels verified (timestamp / label / evidence):
- Visual observation verified or rejected (video time / observation / reviewer):
- Theme/code changed after review (old / new / rationale):
- Negative cases and dissent retained:

## Analytic memo
- Emerging explanation:
- Competing explanation:
- Evidence needed before thesis claim:
- Quote candidates checked against source:
'''
(DRIVE_OUT / "researcher_notes_template.md").write_text(researcher_template, encoding="utf-8")

# DOCX report.
doc = Document()
doc.add_heading("Focus Group Discussion \u2014 Comprehensive Thesis Analysis", 0)
for line in report.splitlines():
    if line.startswith("#"):
        doc.add_heading(line.lstrip("# ").strip(), min(3, len(line) - len(line.lstrip("#"))))
    elif line.strip():
        doc.add_paragraph(line)
doc.add_page_break()
doc.add_heading("Timestamped Transcript", 1)
for line in transcript.splitlines():
    doc.add_paragraph(line)
doc.save(DRIVE_OUT / "FGD_comprehensive_thesis_report.docx")

# Run manifest with hashes (whole-record provenance).
def sha256_of(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for block in iter(lambda: fh.read(65536), b""):
            h.update(block)
    return h.hexdigest()

manifest = {
    "source_file": SOURCE.name,
    "source_sha256": sha256_of(str(SOURCE)),
    "processed_on": __import__("datetime").datetime.now().isoformat(timespec="seconds"),
    "artifacts": {},
}
for p in sorted(DRIVE_OUT.iterdir()):
    if p.is_file() and not p.name.startswith("."):
        manifest["artifacts"][p.name] = sha256_of(str(p))
(DRIVE_OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

zip_path = DRIVE_OUT / "FGD_THESIS_PACKAGE.zip"
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(DRIVE_OUT / "FGD_THESIS_PACKAGE"), "zip", DRIVE_OUT)

print("Package saved on Drive:", DRIVE_OUT)
print(*sorted(p.name for p in DRIVE_OUT.iterdir()), sep="\n")

try:
    files.download(str(zip_path))
except Exception:
    print("Download prompt skipped; the ZIP is already on your Drive.")
    print("Drive location:", zip_path)